# Evaluate Models — merge NEW & OLD test scores with app/target/trade

Builds one eval frame on the **test** samples:
1. `app` -> `ZEST_KEY`, `appDate`
2. `target` -> `ZEST_KEY`, `final_DQ60_m24` (the actual target we model)
3. join app+target on `ZEST_KEY` (inner)
4. **left** join `trade` -> `trade_months_since_oldest_account_opened__all_accounts`, `trade_count__all_accounts`
5. merge in the model scores from `models/model_new/test_scores.parquet` and `models/model_old/test_scores.parquet`.

Run after Build_Model_New / Build_Model_Old have produced `test_scores.parquet`.

In [1]:
import os, sys
import pandas as pd
sys.path.insert(0, os.getcwd())
import configs
from configs import DATA_DIR

TARGET     = 'final_DQ60_m24'
TRADE_COLS = ['trade_months_since_oldest_account_opened__all_accounts',
              'trade_count__all_accounts']
# evaluate on the TEST samples (the rows test_scores covers)
TEST_DIRS  = [os.path.join(DATA_DIR, 'samples', f'{b}_test')
              for b in ['equifax', 'experian', 'transunion']]
MODELS_DIR = os.path.join(DATA_DIR, 'models')
print('test sample dirs:'); [print('  ', d) for d in TEST_DIRS]

test sample dirs:
   /home/jag/payment-processor-research/payment_processing_research_data/samples/equifax_test
   /home/jag/payment-processor-research/payment_processing_research_data/samples/experian_test
   /home/jag/payment-processor-research/payment_processing_research_data/samples/transunion_test


[None, None, None]

In [3]:
# app: ZEST_KEY + appDate
app = pd.concat([pd.read_parquet(os.path.join(d, 'app.parquet'), columns=['ZEST_KEY', 'appDate'])
                 for d in TEST_DIRS], ignore_index=True)

# target: ZEST_KEY + the actual target
tgt = pd.concat([pd.read_parquet(os.path.join(d, 'target.parquet'), columns=['ZEST_KEY', TARGET])
                 for d in TEST_DIRS], ignore_index=True)

base = app.merge(tgt, on='ZEST_KEY', how='inner')

# trade columns (ZEST_KEY is the index in processed_*, reset to a column on read)
def load_trade_cols(variant, cols):
    parts = []
    for d in TEST_DIRS:
        df = pd.read_parquet(os.path.join(d, f'processed_{variant}'), columns=cols)
        parts.append(df.reset_index())   # ZEST_KEY index -> column
    return pd.concat(parts, ignore_index=True)

# 1) the two NON-percent trade cols (identical in new/old) -- read from new, LEFT join
trade = load_trade_cols('new', TRADE_COLS)
base = base.merge(trade, on='ZEST_KEY', how='left')

# 2) the percent feature, which DIFFERS new vs old -- pull from both, suffixed, LEFT join
PCT_COL = 'trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts'
pct_new = load_trade_cols('new', [PCT_COL]).rename(columns={PCT_COL: PCT_COL + '__new'})
pct_old = load_trade_cols('old', [PCT_COL]).rename(columns={PCT_COL: PCT_COL + '__old'})
base = (base.merge(pct_new, on='ZEST_KEY', how='left')
             .merge(pct_old, on='ZEST_KEY', how='left'))

print('base:', base.shape)
print('cols:', list(base.columns))
base.head()

base: (1200000, 7)
cols: ['ZEST_KEY', 'appDate', 'final_DQ60_m24', 'trade_months_since_oldest_account_opened__all_accounts', 'trade_count__all_accounts', 'trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__new', 'trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__old']


,ZEST_KEY,appDate,final_DQ60_m24,trade_months_since_oldest_account_opened__all_accounts,trade_count__all_accounts,trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__new,trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__old
0,00004365121_5,2020-02-28,0.0,135.953510,21.0,0.00000,0.00000
1,00017281785_3,2020-03-15,0.0,NaN,NaN,NaN,NaN
2,00024989447_7,2020-01-23,0.0,130.203906,11.0,0.00000,0.00000
3,00001298985_6,2020-03-07,0.0,292.769872,21.0,0.01087,0.01087
4,00014702927_6,2020-02-03,0.0,452.017495,28.0,0.00000,0.00000


In [4]:
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:.5f}')


In [5]:
base['flg_thin_file'] = (base['trade_months_since_oldest_account_opened__all_accounts'] <= 6) | (base['trade_count__all_accounts'] <= 2)

In [6]:
import numpy as np

mask_missing = (
    base['trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__new'].isna()
    | base['trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__old'].isna()
)

base['file_changed'] = (
    base['trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__new']
    !=
    base['trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__old']
)

base.loc[mask_missing, 'file_changed'] = np.nan

In [7]:
base['file_changed'].value_counts()

file_changed
False    1108475
True       75373
Name: count, dtype: int64

In [8]:
base['file_changed'].value_counts(normalize=True)

file_changed
False   0.93633
True    0.06367
Name: proportion, dtype: float64

In [9]:
base[base['file_changed']==True]

,ZEST_KEY,appDate,final_DQ60_m24,trade_months_since_oldest_account_opened__all_accounts,trade_count__all_accounts,trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__new,trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__old,flg_thin_file,file_changed
45,00009493455_1,2020-01-06,0.00000,126.95127,5.00000,0.05882,0.04348,False,True
46,00030420445_17,2020-01-31,0.00000,53.19206,15.00000,0.00826,0.00395,False,True
48,00028244968_33,2020-03-27,0.00000,160.20041,39.00000,0.00403,0.00281,False,True
67,00026113441_2,2020-02-12,0.00000,122.71299,7.00000,0.16667,0.02174,False,True
76,00001485999_10,2020-01-16,0.00000,122.51586,17.00000,0.01602,0.01449,False,True
...,...,...,...,...,...,...,...,...,...
1199869,12087105_11036815854,2020-01-25,0.00000,196.53792,19.00000,0.03503,0.02688,False,True
1199894,5086160_10978958886,2020-01-27,0.00000,194.13951,10.00000,0.02105,0.01818,False,True
1199954,2220842_11011331872,2020-02-19,0.00000,15.83605,5.00000,0.01667,0.01136,False,True
1199981,20634918_10995968341,2020-01-10,0.00000,38.17738,6.00000,0.01000,0.00870,False,True


In [10]:
# merge in NEW and OLD test scores, and record the ACTUAL prediction column per variant
SCORE_COLS = {}   # variant -> the prediction column name in eval_df

def load_scores(variant):
    p = os.path.join(MODELS_DIR, f'model_{variant}', 'test_scores.parquet')
    s = pd.read_parquet(p)
    if 'ZEST_KEY' not in s.columns:        # scores are usually indexed by ZEST_KEY
        s = s.reset_index()
    assert 'ZEST_KEY' in s.columns, f'no ZEST_KEY in {p} (cols={list(s.columns)})'
    raw_cols = [c for c in s.columns if c != 'ZEST_KEY']
    s = s.rename(columns={c: f'{c}_{variant}' for c in raw_cols})
    suffixed = [f'{c}_{variant}' for c in raw_cols]
    # the actual prediction column: prefer score/pred/prob, else the single score column
    pred = next((c for c in suffixed if any(k in c.lower() for k in ('score', 'pred', 'prob'))),
                suffixed[0])
    SCORE_COLS[variant] = pred
    print(f'{variant}: {len(s):,} rows | score cols {suffixed} | prediction -> {pred}')
    return s

eval_df = (base
           .merge(load_scores('new'), on='ZEST_KEY', how='left')
           .merge(load_scores('old'), on='ZEST_KEY', how='left'))
print('\neval_df:', eval_df.shape, '| SCORE_COLS =', SCORE_COLS)
eval_df.head()

new: 1,200,000 rows | score cols ['final_model_predictions_new'] | prediction -> final_model_predictions_new
old: 1,200,000 rows | score cols ['final_model_predictions_old'] | prediction -> final_model_predictions_old

eval_df: (1200000, 11) | SCORE_COLS = {'new': 'final_model_predictions_new', 'old': 'final_model_predictions_old'}


,ZEST_KEY,appDate,final_DQ60_m24,trade_months_since_oldest_account_opened__all_accounts,trade_count__all_accounts,trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__new,trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__old,flg_thin_file,file_changed,final_model_predictions_new,final_model_predictions_old
0,00004365121_5,2020-02-28,0.00000,135.95351,21.00000,0.00000,0.00000,False,False,0.09164,0.09155
1,00017281785_3,2020-03-15,0.00000,NaN,NaN,NaN,NaN,False,NaN,0.36232,0.36185
2,00024989447_7,2020-01-23,0.00000,130.20391,11.00000,0.00000,0.00000,False,False,0.07122,0.06914
3,00001298985_6,2020-03-07,0.00000,292.76987,21.00000,0.01087,0.01087,False,False,0.13159,0.12696
4,00014702927_6,2020-02-03,0.00000,452.01750,28.00000,0.00000,0.00000,False,False,0.02532,0.02466


In [11]:
eval_df['final_DQ60_m24'].sum()

85140.0

In [12]:
# AUC comparison using the ACTUAL prediction column from test_scores (SCORE_COLS)
from sklearn.metrics import roc_auc_score
y = eval_df[TARGET]
for variant in ['new', 'old']:
    col = SCORE_COLS[variant]
    m = y.notna() & eval_df[col].notna()
    try:
        print(f'{variant}: AUC = {roc_auc_score(y[m], eval_df.loc[m, col]):.4f}  '
              f'(col={col}, n={m.sum():,})')
    except Exception as e:
        print(f'{variant}: AUC failed for {col}: {e}')

new: AUC = 0.7763  (col=final_model_predictions_new, n=1,200,000)
old: AUC = 0.7764  (col=final_model_predictions_old, n=1,200,000)


In [13]:
new_feature_importance = pd.read_parquet('~/payment-processor-research/payment_processing_research_data/models/model_new/feature_importance.parquet')

In [14]:
from sklearn.metrics import roc_auc_score

def load_scores(variant):
  p = os.path.join(MODELS_DIR, f'model_{variant}', 'test_scores.parquet')
  s = pd.read_parquet(p)
  if 'ZEST_KEY' not in s.columns:        # scores are usually indexed by ZEST_KEY
      s = s.reset_index()
  raw_cols = [c for c in s.columns if c != 'ZEST_KEY']
  s = s.rename(columns={c: f'{c}_{variant}' for c in raw_cols})
  suffixed = [f'{c}_{variant}' for c in raw_cols]
  pred = next((c for c in suffixed if any(k in c.lower() for k in ('score', 'pred', 'prob'))),
              suffixed[0])
  return s, pred

s_new, pred_new = load_scores('new')
s_old, pred_old = load_scores('old')
scored = base.merge(s_new, on='ZEST_KEY', how='left').merge(s_old, on='ZEST_KEY', how='left')

y = scored[TARGET]
for variant, col in [('new', pred_new), ('old', pred_old)]:
  m = y.notna() & scored[col].notna()
  print(f'{variant}: AUC = {roc_auc_score(y[m], scored.loc[m, col]):.4f}  (col={col}, n={m.sum():,})')

new: AUC = 0.7763  (col=final_model_predictions_new, n=1,200,000)
old: AUC = 0.7764  (col=final_model_predictions_old, n=1,200,000)


In [15]:
scored_thin_file = scored[scored['flg_thin_file']==True]
y = scored_thin_file[TARGET]
for variant, col in [('new', pred_new), ('old', pred_old)]:
  m = y.notna() & scored_thin_file[col].notna()
  print(f'{variant}: AUC = {roc_auc_score(y[m], scored_thin_file.loc[m, col]):.4f}  (col={col}, n={m.sum():,})')

new: AUC = 0.6760  (col=final_model_predictions_new, n=66,089)
old: AUC = 0.6765  (col=final_model_predictions_old, n=66,089)


In [16]:
scored_file_changed = scored[scored['file_changed']==True]
y = scored_file_changed[TARGET]
for variant, col in [('new', pred_new), ('old', pred_old)]:
  m = y.notna() & scored_file_changed[col].notna()
  print(f'{variant}: AUC = {roc_auc_score(y[m], scored_file_changed.loc[m, col]):.4f}  (col={col}, n={m.sum():,})')

new: AUC = 0.6924  (col=final_model_predictions_new, n=75,373)
old: AUC = 0.6926  (col=final_model_predictions_old, n=75,373)


In [28]:
MODELS = os.path.expanduser('~/payment-processor-research/payment_processing_research_data/models')
new = pd.read_parquet(os.path.join(MODELS, 'model_new', 'test_fe_data.parquet'))
old = pd.read_parquet(os.path.join(MODELS, 'model_old', 'test_fe_data.parquet'))


In [30]:
for feature in new.columns:
    print(feature)

ZEST_KEY
trade_max_percent_of_DQ60_in_last_24_months__all_open_installment
trade_min_percent_of_DQ30_or_greater_in_last_6_months__with_recent_payment_open_accounts
trade_min_percent_of_DQ30_or_greater_in_last_6_months__with_utilization_over_25_percent_open_revolving
trade_min_percent_of_DQ60_in_last_12_months__recently_opened_open_cc
trade_max_percent_of_DQ60_in_last_12_months__active_open_auto
trade_min_percent_of_DQ30_in_last_24_months__non_derog_open_auto
trade_max_percent_of_DQ60_in_last_12_months__all_open_auto
trade_max_percent_of_DQ30_or_greater_in_last_6_months__joint_open_charge_card
trade_mean_percent_of_DQ30_in_last_24_months__derog_open_accounts
trade_min_percent_of_DQ30_in_last_24_months__non_derog_open_installment
trade_min_percent_of_DQ30_in_last_24_months__non_derog_open_revolving
trade_max_percent_of_DQ30_or_greater_in_last_6_months__active_open_revolving
trade_min_percent_of_DQ30_in_last_12_months__joint_open_unsecure_personal
trade_mean_percent_of_DQ30_or_greater_in_

In [ ]:

print('new', new.shape, '| old', old.shape)
print('cols only in new:', list(set(new.columns) - set(old.columns))[:10])
print('cols only in old:', list(set(old.columns) - set(new.columns))[:10])

# align on common rows (index = ZEST_KEY) + common columns, same order
idx  = new.index.intersection(old.index)
cols = [c for c in new.columns if c in old.columns]
new  = new.loc[idx, cols].sort_index()
old  = old.loc[idx, cols].sort_index()
print('aligned:', new.shape)

# per-column change
rows = []
for c in cols:
  a, b = new[c], old[c]
  if a.dtype.kind in 'fiub' and b.dtype.kind in 'fiub':
      af, bf = a.astype('float64'), b.astype('float64')
      neq = ~np.isclose(af, bf, equal_nan=True)
      mad = float(np.abs(af - bf).mean())
  else:
      neq = a.astype('string').fillna('∅') != b.astype('string').fillna('∅')
      mad = np.nan
  n = int(neq.sum())
  if n:
      rows.append({'col': c, 'n_changed': n, 'pct_rows': 100*n/len(new), 'mean_abs_diff': mad})

diff = pd.DataFrame(rows).sort_values('n_changed', ascending=False)
ncells = len(new) * len(cols)
print(f'\n{len(diff)} of {len(cols)} columns differ '
    f'| {diff["n_changed"].sum():,} of {ncells:,} cells changed '
    f'({100*diff["n_changed"].sum()/ncells:.3f}%)')
print('\nmost-changed columns:')
print(diff.head(30).to_string(index=False))

In [17]:
import os, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import roc_auc_score

MODELS = os.path.expanduser('~/payment-processor-research/payment_processing_research_data/models')
TARGET = 'final_DQ60_m24'

# ---- load FE matrices (NEW vs OLD), align ----
new = pd.read_parquet(os.path.join(MODELS, 'model_new', 'test_fe_data.parquet'))
old = pd.read_parquet(os.path.join(MODELS, 'model_old', 'test_fe_data.parquet'))
cols = [c for c in new.columns if c in old.columns]
new, old = new[cols], old.loc[new.index, cols]

# numeric matrix, fill missing
X = new.select_dtypes('number').fillna(-1.0).astype('float32')
num_cols = X.columns

# per-row total drift between NEW and OLD (how much the fix moved this applicant)
drift = X.values - old[num_cols].fillna(-1.0).astype('float32').values
row_drift = np.abs(drift).sum(axis=1)

# ---- PCA -> 5 components, then KMeans -> 5 clusters ----
Xs  = StandardScaler().fit_transform(X.values)
pcs = PCA(n_components=5, random_state=0).fit_transform(Xs)

In [19]:
km  = MiniBatchKMeans(n_clusters=5, random_state=0, n_init=10, batch_size=10000)

In [20]:

cluster = km.fit_predict(pcs)

OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.


In [21]:
prof = pd.DataFrame({'cluster': cluster, 'row_drift': row_drift})
prof['ZEST_KEY'] = new['ZEST_KEY'].values        # ZEST_KEY is a column; same order as prof

In [22]:
print('cluster sizes:'); print(prof['cluster'].value_counts().sort_index())
print('\nmean absolute NEW-vs-OLD drift per cluster:')

cluster sizes:
cluster
0    250705
1    310129
2    221331
3    184839
4    232996
Name: count, dtype: int64

mean absolute NEW-vs-OLD drift per cluster:


In [23]:

print(prof.groupby('cluster')['row_drift'].agg(['mean','median','max']).round(4))
print('\nfraction of each cluster that changed at all:')
print(prof.assign(changed=prof['row_drift']>0).groupby('cluster')['changed'].mean().round(4))

           mean  median        max
cluster                           
0       0.41480 0.00000  409.62500
1       0.32770 0.00000  720.95239
2       0.67100 0.00000  432.00000
3       1.01160 0.00000 1080.00000
4       0.26370 0.00000  420.00000

fraction of each cluster that changed at all:
cluster
0   0.08940
1   0.05640
2   0.11860
3   0.11050
4   0.07060
Name: changed, dtype: float64


In [24]:

# ---- target + both predictions, merged on the ZEST_KEY COLUMN (not the index) ----
TEST_DIRS = [os.path.join(os.path.dirname(MODELS), 'samples', f'{b}_test')
           for b in ['equifax', 'experian', 'transunion']]
tgt = pd.concat([pd.read_parquet(os.path.join(d,'target.parquet'), columns=['ZEST_KEY',TARGET])
               for d in TEST_DIRS], ignore_index=True)


In [25]:
def load_pred(variant):
  s = pd.read_parquet(os.path.join(MODELS, f'model_{variant}', 'test_scores.parquet'))
  if 'ZEST_KEY' not in s.columns:
      s = s.reset_index()
  pcol = [c for c in s.columns if c != 'ZEST_KEY'][0]
  return s[['ZEST_KEY', pcol]].rename(columns={pcol: f'pred_{variant}'})

In [26]:
prof = (prof
      .merge(tgt,             on='ZEST_KEY', how='left')
      .merge(load_pred('new'), on='ZEST_KEY', how='left')
      .merge(load_pred('old'), on='ZEST_KEY', how='left'))

In [27]:

print('\nmatched targets:', prof[TARGET].notna().sum(), 'of', len(prof))   # expect ~1.2M

print('\nper-cluster target rate, drift, and AUC new vs old:')
for c, g in prof.groupby('cluster'):
  m = g[TARGET].notna() & g['pred_new'].notna()
  auc_n = roc_auc_score(g.loc[m,TARGET], g.loc[m,'pred_new']) if m.sum() else float('nan')
  auc_o = roc_auc_score(g.loc[m,TARGET], g.loc[m,'pred_old']) if m.sum() else float('nan')
  print(f'cluster {c}: n={len(g):>7,} | bad_rate={g[TARGET].mean():.4f} '
        f'| mean_drift={g["row_drift"].mean():.4f} | AUC new={auc_n:.4f} old={auc_o:.4f}')


matched targets: 1200000 of 1200000

per-cluster target rate, drift, and AUC new vs old:
cluster 0: n=250,705 | bad_rate=0.0551 | mean_drift=0.4148 | AUC new=0.7515 old=0.7516
cluster 1: n=310,129 | bad_rate=0.0534 | mean_drift=0.3277 | AUC new=0.7860 old=0.7862
cluster 2: n=221,331 | bad_rate=0.0684 | mean_drift=0.6710 | AUC new=0.6938 old=0.6941
cluster 3: n=184,839 | bad_rate=0.1730 | mean_drift=1.0116 | AUC new=0.6410 old=0.6409
cluster 4: n=232,996 | bad_rate=0.0329 | mean_drift=0.2637 | AUC new=0.7829 old=0.7830
